# Final convergence check --- scaling verification

`scalings.py` holds the transforms. This notebook drives and plots them;
`run_scan.py` imports the same functions, so a terminal run and a plot here
cannot disagree about what a scaling is.

| kind | transform | `te` | `ne` |
|------|-----------|------|------|
| `omt_omne` | gradient power law about `rhot_midped` | Te, Ti, Tz | ne (ni, nz follow) |
| `mtanh_full` | Stefanikova F_full, `b_height` scaled | Te only | ne (ni, nz follow) |

Legends come from `describe()`, which reads the transform's own record ---
`alpha` for omt/omne, `var` and `scale_height` for mtanh_full --- so a curve
cannot be mislabelled. `ratios()` prints what the profile actually did, since
neither knob is a value ratio and neither can be read off by eye.


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from scalings import SHOTS, load, scale, run, compare, ratios, describe, tag


## Case


In [ ]:
SHOT   = 129015
SCALES = (0.7, 1.3)
PED    = (0.8, 1.0)   # pedestal window for the zoomed panels

base = load(SHOT)


## 1. `mtanh_full` --- the Hatch handoff transform

`scale_height` multiplies `b_height` alone. `b_sol` and the Gaussian core term
are held, so the achieved ratio is pulled toward 1 --- by how much is a
per-discharge property, not a constant. Read the table, not the requested number.


In [ ]:
te_full = [scale(base, 'mtanh_full', te=s) for s in SCALES]
ne_full = [scale(base, 'mtanh_full', ne=s) for s in SCALES]

ratios(base, *te_full, var='Te')
print()
ratios(base, *ne_full, var='ne')


In [ ]:
compare(base, *te_full, xlim=PED)
compare(base, *ne_full, xlim=PED)


## 2. `omt_omne` --- gradient scaling

`alpha` is the asymptotic exponent, not a value ratio and not the achieved
gradient ratio. Inside the ramp the two differ and can invert in sign for
`alpha < 1`.


In [ ]:
te_grad = [scale(base, 'omt_omne', te=s) for s in SCALES]
ne_grad = [scale(base, 'omt_omne', ne=s) for s in SCALES]

ratios(base, *te_grad, var='Te')
print()
ratios(base, *ne_grad, var='ne')


In [ ]:
compare(base, *te_grad, xlim=PED)
compare(base, *ne_grad, xlim=PED)


## 3. Same number, two transforms

The two agree on nothing but the label. Read before quoting a scan point.


In [ ]:
both = [scale(base, k, te=s) for s in SCALES for k in ('mtanh_full', 'omt_omne')]

ratios(base, *both, var='Te')
compare(base, *both, xlim=PED)
compare(base, *both)


## 4. Identity check

`omt_omne` at 1.0 returns the source profile. `mtanh_full` returns its fit,
so the gap below is the fit residual --- and it rides on every `mtanh_full`
case, including the ones in the Hatch handoff package.


In [ ]:
ident = scale(base, 'mtanh_full', te=1.0 + 1e-12, ne=1.0 + 1e-12)

ratios(base, ident, var='Te')
ratios(base, ident, var='ne')
compare(base, ident, labels=['base (data)', 'mtanh_full 1.0 (fit)'], xlim=PED)


## 5. Hand off to cheaseBS

Minutes per case --- run it in a terminal, not here:

```bash
python run_scan.py --shot 129015 --kind mtanh_full --scan te 0.7 1.3 --savedir out/
```

`run(..., gfile=False)` is the notebook path and writes nothing.


In [ ]:
# run(SHOT, 'mtanh_full', te=1.3, gfile=True, savedir='out/')
